In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from concurrent.futures import ThreadPoolExecutor, as_completed

from nn.metrics import mse
from nn.wrappers import MinMaxScaler, Identity
from nn.network import Network
from nn.layer import Layer
from nn.activation_functions import *

In [2]:
def visualize_convergence(train_path, test_path, models, lrs, epochs_list,
                          batch_size=32,
                          normalize=False,
                          mode="momentum",
                          momentum=0.9,
                          decay_rate=0.9,
                          l2_lambda=0.001,
                          patience=20,
                          random_seed_list=[42]):

    colors = cm.get_cmap('tab10', len(random_seed_list))

    # 1. Wczytanie danych
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)

    X_train_orig = df_train.iloc[:, :-1].values.astype(np.float32)
    y_train_orig = df_train.iloc[:, -1:].values.astype(np.float32)
    X_test_orig = df_test.iloc[:, :-1].values.astype(np.float32)
    y_test_orig = df_test.iloc[:, -1:].values.astype(np.float32)

    X_train, y_train = X_train_orig.copy(), y_train_orig.copy()
    X_test, y_test = X_test_orig.copy(), y_test_orig.copy()

    scaler_y = None

    # 2. Skalowanie
    if normalize:
        scaler_X = MinMaxScaler()
        scaler_y = MinMaxScaler()
        X_train = scaler_X.fit_transform(X_train)
        X_test = scaler_X.transform(X_test)
        y_train = scaler_y.fit_transform(y_train)
        y_test = scaler_y.transform(y_test)

    epochs_sorted = sorted(epochs_list)
    x_plot_train = X_train_orig[:, 0]
    x_plot_test = X_test_orig[:, 0]

    # Słownik do grupowania wyników
    grouped_results = {}

    print(f"[{pd.Timestamp.now().strftime('%H:%M:%S')}] Rozpoczynamy trening...\n")

    # ==========================================
    # ETAP 1: GŁÓWNA PĘTLA TRENINGOWA
    # ==========================================
    max_epochs = max(epochs_sorted)

    for model_idx, base_model in enumerate(models):
        for lr in lrs:
            key = (model_idx, lr)
            grouped_results[key] = []

            for seed_idx, seed in enumerate(random_seed_list):
                np.random.seed(seed)
                model = base_model()

                full_loss_history = []
                full_val_loss_history = []
                predictions_log = {}

                current_epoch = 0
                stopped_early = False

                desc_str = f"M{model_idx+1} | LR={lr} | Seed={seed}"

                for ep in epochs_sorted:
                    epochs_to_run = ep - current_epoch

                    if epochs_to_run > 0 and not stopped_early:
                        val_data = (X_test, y_test) if patience is not None else None

                        hist = model.fit_with_loss_history(
                            X_train, y_train,
                            batch_size=batch_size,
                            epochs=epochs_to_run,
                            learning_rate=lr,
                            mode=mode,
                            momentum=momentum,
                            decay_rate=decay_rate,
                            l2_lambda=l2_lambda,
                            validation_data=val_data,
                            patience=patience,
                            tqdm_desc=desc_str,
                            total_epochs=max_epochs
                        )

                        full_loss_history.extend(hist['loss'])
                        if 'val_loss' in hist:
                            full_val_loss_history.extend(hist['val_loss'])

                        actual_epochs_run = len(hist['loss'])
                        current_epoch += actual_epochs_run

                        if actual_epochs_run < epochs_to_run:
                            stopped_early = True
                    else:
                        current_epoch = ep

                    # Zapisywanie predykcji
                    pred_train_norm = model.predict(X_train)
                    pred_test_norm = model.predict(X_test)

                    if normalize and scaler_y is not None:
                        pred_train_unnorm = scaler_y.inverse_transform(pred_train_norm)
                        pred_test_unnorm = scaler_y.inverse_transform(pred_test_norm)
                    else:
                        pred_train_unnorm = pred_train_norm
                        pred_test_unnorm = pred_test_norm

                    train_mse = mse(y_train_orig, pred_train_unnorm)
                    test_mse = mse(y_test_orig, pred_test_unnorm)
                    predictions_log[ep] = (pred_train_unnorm, pred_test_unnorm, train_mse, test_mse)

                grouped_results[key].append({
                    'seed': seed,
                    'seed_idx': seed_idx,
                    't_hist': full_loss_history,
                    'v_hist': full_val_loss_history,
                    'p_log': predictions_log
                })

    # ==========================================
    # ETAP 2: GENEROWANIE WYKRESÓW
    # ==========================================
    print(f"\n[{pd.Timestamp.now().strftime('%H:%M:%S')}] Trening zakończony! Generowanie wykresów...")

    for key in sorted(grouped_results.keys()):
        model_idx, lr = key
        group = sorted(grouped_results[key], key=lambda x: x['seed_idx'])

        print(f"\n{'='*60}")
        print(f"MODEL {model_idx + 1} | LR: {lr}")
        print(f"{'='*60}")

        fig_loss, ax_loss = plt.subplots(figsize=(10, 4))
        ax_loss.set_title(f"Model {model_idx + 1} | LR={lr} | Historia Błędu")
        ax_loss.set_xlabel("Faktyczna liczba ukończonych Epok")
        ax_loss.set_ylabel("MSE Loss")
        ax_loss.grid(True, alpha=0.3)

        num_cols = len(epochs_sorted)
        fig_scatter, axes = plt.subplots(2, num_cols, figsize=(5 * num_cols, 8))
        if num_cols == 1: axes = np.array([[axes[0]], [axes[1]]])

        for i, ep in enumerate(epochs_sorted):
            axes[0, i].scatter(x_plot_train, y_train_orig, alpha=0.2, color='black', marker='x')
            axes[0, i].set_title(f"Trening | Zrzut z epoki: {ep}", fontsize=10, fontweight='bold')
            axes[0, i].grid(True, alpha=0.3)

            axes[1, i].scatter(x_plot_test, y_test_orig, alpha=0.2, color='black', marker='x')
            axes[1, i].set_title(f"Test | Zrzut z epoki: {ep}", fontsize=10, fontweight='bold')
            axes[1, i].grid(True, alpha=0.3)

        for res in group:
            seed = res['seed']
            seed_idx = res['seed_idx']
            t_hist = res['t_hist']
            v_hist = res['v_hist']
            p_log = res['p_log']
            color = colors(seed_idx)

            for i, ep in enumerate(epochs_sorted):
                p_train, p_test, tr_mse, te_mse = p_log[ep]
                axes[0, i].scatter(x_plot_train, p_train, color=color, s=10, alpha=0.6, label=f'S{seed} MSE:{tr_mse:.3f}')
                axes[1, i].scatter(x_plot_test, p_test, color=color, s=10, alpha=0.6, label=f'S{seed} MSE:{te_mse:.3f}')

            epochs_x = range(1, len(t_hist) + 1)
            ax_loss.plot(epochs_x, t_hist, color=color, label=f'Seed {seed} (Train)')
            if v_hist:
                ax_loss.plot(epochs_x, v_hist, color=color, linestyle='--', alpha=0.6, label=f'Seed {seed} (Val)')

        ax_loss.legend(loc='upper right')
        for i in range(num_cols):
            axes[0, i].legend(fontsize='small')
            axes[1, i].legend(fontsize='small')

        fig_scatter.tight_layout()
        plt.show()

In [ ]:
# ==========================================
# TEST 3: Funkcja aktywacji TANH
# ==========================================

lista_modeli_tanh = [
    # 1 warstwa ukryta (50 neuronów)
    lambda: Network([Layer(1, 50, Tanh()),
                     Layer(50, 1, Sigmoid())], task="regression"),

    # 2 warstwy ukryte (50 + 25 neuronów)
    lambda: Network([Layer(1, 50, Tanh()),
                     Layer(50, 25, Tanh()),
                     Layer(25, 1, Sigmoid())], task="regression"),

    # 3 warstwy ukryte (50 + 25 + 20 neuronów)
    lambda: Network([Layer(1, 50, Tanh()),
                     Layer(50, 25, Tanh()),
                     Layer(25, 20, Tanh()),
                     Layer(20, 1, Sigmoid())], task="regression")
]

lista_lr = [0.1, 0.001]
lista_epok = [100, 500, 1000]

print("ROZPOCZĘCIE TESTÓW DLA FUNKCJI TANH")
visualize_convergence('datasets/regression/multimodal-large-training.csv',
                      'datasets/regression/multimodal-large-test.csv',
                      lista_modeli_tanh,
                      lista_lr,
                      lista_epok,
                      normalize=True,
                      random_seed_list=[3, 1, 2005])

ROZPOCZĘCIE TESTÓW DLA FUNKCJI TANH
[20:21:04] Rozpoczynamy trening...



M1 | LR=0.1 | Seed=3:   0%|          | 0/1000 [00:00<?, ?it/s]


[Early Stopping] Aktywowano! Najlepszy błąd walidacji: 0.063492


M1 | LR=0.1 | Seed=1:   0%|          | 0/1000 [00:00<?, ?it/s]


[Early Stopping] Aktywowano! Najlepszy błąd walidacji: 0.063338


M1 | LR=0.1 | Seed=2005:   0%|          | 0/1000 [00:00<?, ?it/s]


[Early Stopping] Aktywowano! Najlepszy błąd walidacji: 0.063605


M1 | LR=0.001 | Seed=3:   0%|          | 0/1000 [00:00<?, ?it/s]

M1 | LR=0.001 | Seed=1:   0%|          | 0/1000 [00:00<?, ?it/s]